# Anchorage Harbor: ConvLSTM with drivers

Do the outside drivers (tides, river flow, weather) help the ConvLSTM forecast the depth map ~4 weeks ahead?

To find out, we train the **same model twice**: once with the depth maps only (as in `anchorage_convlstm.ipynb`),
once with the drivers added, each with the same 3 random seeds. Then we compare both with the baselines.

1. Load `timeseries_v2`
2. Prepare the maps (same as before)
3. Prepare the drivers
4. The model: ConvLSTM, now with drivers
5. Training
6. Train both versions
7. Compare

## 1. Load

Same as before, plus the drivers from `timeseries_v2` (`anchorage_timeseries_v2.ipynb`):
- per map (697): `<driver>_before` = the 7 days before that map's survey (mean, or total for rain and snow)
- per sample (515): `tide_range_ahead` = mean daily tidal range from the last input to the target

In [ ]:
import copy
import time

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/ehydro/anchorage'
except ImportError:              # not in Colab: the data folder next to the notebook
    DATA_DIR = 'data'

d = np.load(f'{DATA_DIR}/timeseries_v2/timeseries.npz')
maps, windows, masks, split = d['maps'], d['windows'], d['masks'], d['split']
dates, dredged = d['dates'], d['dredged']

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('maps', maps.shape, ' windows', windows.shape, ' device', device)
for name in ['train', 'val', 'test']:
    print(f'  {name}: {(split == name).sum()} samples')

## 2. Prepare the maps

Exactly as in `anchorage_convlstm.ipynb`:
- **inputs**: each of the 4 input maps becomes 2 channels, the normalised depth (0 where not measured) and a "measured" flag
- **target**: the **change** from the last input to the target, divided by its training standard deviation,
  0 outside the mask

In [ ]:
inputs = maps[windows[:, :-1]]              # (samples, 4, H, W)
target = maps[windows[:, -1]]               # (samples, H, W)
last = inputs[:, -1]                        # the last input map
train = split == 'train'

MEAN, STD = np.nanmean(inputs[train]), np.nanstd(inputs[train])
measured = ~np.isnan(inputs)
x = np.stack([np.where(measured, (inputs - MEAN) / STD, 0), measured], axis=2).astype(np.float32)

change = np.where(masks, target - last, 0)                   # ft, 0 outside the mask
CHANGE_STD = change[train][masks[train]].std()
y = (change / CHANGE_STD).astype(np.float32)

print('x', x.shape, ' y', y.shape, f' change std {CHANGE_STD:.2f} ft')

## 3. Prepare the drivers

**Per input map: 6 numbers.** For each of the 4 input maps we take its 6 `_before` values. `before[windows[:, :-1]]`
works exactly like `maps[windows[:, :-1]]`: the shelf numbers fetch the rows that belong to the input maps.
That gives `drv` with shape (samples, 4, 6).

**Per sample: 1 number.** `tide_range_ahead`, the tide during the 4 weeks being predicted. Shape (samples, 1).

Before the model can use them:
- **log** for river flow, rain and snow: they are very skewed (Knik goes from ~1,000 to ~40,000 cfs, most weeks have
  little rain and a few have a lot). `log1p(v) = log(1 + v)` squeezes the big values together and keeps 0 at 0.
- **normalise** each driver with its training mean and standard deviation, so all of them are around ±1
- **missing values** (Knik at the edge of the season, 17 maps) become 0 after normalising, which means "the training average"

**We leave out snowfall.** In the training inputs it is almost always 0 (standard deviation 0.02 after the log), so a
single snowy week in val or test would become a value of ~38 after normalising, far outside anything the model has seen.
There is too little snow during our surveys for the model to learn anything from it.

In [ ]:
DRIVERS = ['tide_range', 'knik_flow', 'matanuska_flow', 'shipcreek_flow', 'rain', 'temp']    # no snowfall, see above
LOG = ['knik_flow', 'matanuska_flow', 'shipcreek_flow', 'rain']

before = np.stack([d[f'{name}_before'] for name in DRIVERS], axis=1)     # (697, 6): one row per map
for j, name in enumerate(DRIVERS):
    if name in LOG:
        before[:, j] = np.log1p(before[:, j])

drv = before[windows[:, :-1]]                                  # (samples, 4, 6)
D_MEAN = np.nanmean(drv[train], axis=(0, 1))                   # one mean per driver
D_STD = np.nanstd(drv[train], axis=(0, 1))
drv = np.nan_to_num((drv - D_MEAN) / D_STD).astype(np.float32)   # NaN -> 0 = training average

ahead = d['tide_range_ahead'][:, None]                         # (samples, 1)
ahead = ((ahead - ahead[train].mean()) / ahead[train].std()).astype(np.float32)

print('drv', drv.shape, ' ahead', ahead.shape)
print('training mean / std per driver (after log):')
for name, mu, sd in zip(DRIVERS, D_MEAN, D_STD):
    print(f'  {name:15} {mu:8.2f} {sd:7.2f}')

x, y, m = torch.from_numpy(x), torch.from_numpy(y), torch.from_numpy(masks.astype(np.float32))
drv, ahead = torch.from_numpy(drv), torch.from_numpy(ahead)
print('any NaN left:', bool(torch.isnan(x).any() or torch.isnan(y).any() or torch.isnan(drv).any() or torch.isnan(ahead).any()))

## 4. The model

The same ConvLSTM as before. The drivers are single numbers, not maps, so we turn each one into a **constant map**
(the same value on every square) and add it as extra channels:

- **per input step**: the 6 drivers of that input map go in next to its 2 depth channels → 8 channels per step
- **at the end**: the tide ahead goes in next to the memory `h`, right before the head, since it is about the future
  period, not about any input map

`as_maps` does the "constant map" part: `v[:, :, None, None]` adds two empty dimensions, and `expand` repeats the
value over H × W without copying memory.

`n_drivers` and `n_ahead` say how many of these the model gets. With `n_drivers=0, n_ahead=0`
it is exactly the old model.

In [ ]:
def as_maps(v, H, W):
    """(batch, n) numbers -> (batch, n, H, W) constant maps."""
    return v[:, :, None, None].expand(-1, -1, H, W)


class ConvLSTMCell(nn.Module):
    def __init__(self, in_channels, hidden):
        super().__init__()
        self.hidden = hidden
        self.conv = nn.Conv2d(in_channels + hidden, 4 * hidden, kernel_size=3, padding=1)

    def forward(self, x, h, c):
        i, f, o, g = self.conv(torch.cat([x, h], dim=1)).chunk(4, dim=1)
        c = torch.sigmoid(f) * c + torch.sigmoid(i) * torch.tanh(g)
        h = torch.sigmoid(o) * torch.tanh(c)
        return h, c


class ConvLSTM(nn.Module):
    def __init__(self, n_drivers=0, n_ahead=0, hidden=32):
        super().__init__()
        self.n_drivers, self.n_ahead = n_drivers, n_ahead
        self.cell = ConvLSTMCell(2 + n_drivers, hidden)
        self.head = nn.Conv2d(hidden + n_ahead, 1, kernel_size=1)

    def forward(self, x, drv, ahead):          # x (batch, time, 2, H, W), drv (batch, time, 6), ahead (batch, 1)
        batch, steps, _, H, W = x.shape
        drv, ahead = drv[:, :, :self.n_drivers], ahead[:, :self.n_ahead]     # keep only what this model uses
        h = torch.zeros(batch, self.cell.hidden, H, W, device=x.device)
        c = torch.zeros_like(h)
        for t in range(steps):
            step_input = torch.cat([x[:, t], as_maps(drv[:, t], H, W)], dim=1)
            h, c = self.cell(step_input, h, c)
        return self.head(torch.cat([h, as_maps(ahead, H, W)], dim=1))[:, 0]   # (batch, H, W): predicted change

## 5. Training

Same as before: masked MSE loss, Adam, batch 8, 30 epochs, and the weights of the epoch with the best
validation MAE are kept. Packed into a function `train_model`, so we can run it for both versions and several seeds.

In [ ]:
EPOCHS = 30
BATCH = 8

train_idx = np.where(split == 'train')[0]
val_idx = np.where(split == 'val')[0]
test_idx = np.where(split == 'test')[0]

def masked_mse(pred, truth, mask):
    return ((pred - truth) ** 2 * mask).sum() / mask.sum()

def predict(model, indices):
    """Predicted change in ft for the given samples."""
    model.eval()
    out = []
    with torch.no_grad():
        for start in range(0, len(indices), BATCH):
            b = indices[start:start + BATCH]
            out.append(model(x[b].to(device), drv[b].to(device), ahead[b].to(device)).cpu())
    return torch.cat(out).numpy() * CHANGE_STD

def mae(pred_change, indices):
    """Mean absolute error in ft over all masked squares of these samples."""
    return np.abs(pred_change - change[indices])[masks[indices]].mean()

def train_model(n_drivers, n_ahead, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = ConvLSTM(n_drivers, n_ahead).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    best_mae, best_weights = np.inf, None
    for epoch in range(EPOCHS):
        model.train()
        order = np.random.permutation(train_idx)
        for start in range(0, len(order), BATCH):
            b = order[start:start + BATCH]
            pred = model(x[b].to(device), drv[b].to(device), ahead[b].to(device))
            loss = masked_mse(pred, y[b].to(device), m[b].to(device))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        val_mae = mae(predict(model, val_idx), val_idx)
        if val_mae < best_mae:
            best_mae, best_weights = val_mae, copy.deepcopy(model.state_dict())

    model.load_state_dict(best_weights)
    return model

## 6. Train both versions

- **maps only**: `n_drivers=0, n_ahead=0`, the old model
- **maps + drivers**: all 6 drivers per input step and the tide ahead

Each with seeds 0, 1 and 2. One run gave quite different numbers from the next in the first notebook
(val 0.778 vs 0.726), so a single run can't tell a real improvement from luck. 6 runs × 30 epochs take a few minutes on a GPU.

In [ ]:
VERSIONS = {'maps only': (0, 0), 'maps + drivers': (len(DRIVERS), 1)}
SEEDS = [0, 1, 2]

results = {}          # (version, seed) -> (val MAE, test MAE)
models = {}
for version, (n_drivers, n_ahead) in VERSIONS.items():
    for seed in SEEDS:
        start_time = time.time()
        model = train_model(n_drivers, n_ahead, seed)
        results[version, seed] = (mae(predict(model, val_idx), val_idx), mae(predict(model, test_idx), test_idx))
        models[version, seed] = model
        print(f'{version:15} seed {seed}  val {results[version, seed][0]:.3f} ft  '
              f'test {results[version, seed][1]:.3f} ft  ({time.time() - start_time:.0f} s)')

## 7. Compare

Baselines as before: **nothing changes** (change = 0) and **shoaling rate** (each square keeps its average
training rate). For the ConvLSTM versions: the mean over the 3 seeds, and the spread (lowest–highest).

In [ ]:
days = dates.astype('datetime64[D]').astype('int64')        # date as a day number
year = dates.astype('datetime64[Y]').astype(int) + 1970

total_change = np.zeros(maps.shape[1:])
total_days = np.zeros(maps.shape[1:])
for p in range(len(maps) - 1):
    same_season = days[p + 1] - days[p] < 60
    if same_season and year[p + 1] <= 2022:
        step = maps[p + 1] - maps[p]
        ok = ~np.isnan(step) & ~dredged[p + 1]
        total_change[ok] += step[ok]
        total_days[ok] += days[p + 1] - days[p]
rate = np.where(total_days > 0, total_change / np.maximum(total_days, 1), 0)    # ft per day

def rate_prediction(indices):
    gap = days[windows[indices, -1]] - days[windows[indices, -2]]               # days from last input to target
    return rate[None] * gap[:, None, None]

rows = []
for name, idx in [('val', val_idx), ('test', test_idx)]:
    base = mae(np.zeros_like(change[idx]), idx)
    rows.append((name, 'nothing changes', base, base, base))
    shoal = mae(rate_prediction(idx), idx)
    rows.append((name, 'shoaling rate', shoal, shoal, shoal))
    col = 0 if name == 'val' else 1
    for version in VERSIONS:
        runs = [results[version, s][col] for s in SEEDS]
        rows.append((name, version, np.mean(runs), min(runs), max(runs)))

print(f'{"":6} {"":16} {"MAE":>8} {"spread":>15} {"vs nothing changes":>20}')
for name, label, mean, low, high in rows:
    base = next(r[2] for r in rows if r[0] == name and r[1] == 'nothing changes')
    spread = f'{low:.3f}–{high:.3f}' if high > low else ''
    print(f'{name:6} {label:16} {mean:6.3f} ft {spread:>15} {100 * (1 - mean / base):17.0f} %')

**How to read it:** the drivers help only if "maps + drivers" beats "maps only" on **both** val and test, by more
than the spread between seeds. If the ranges overlap, the difference may just be luck.

One test sample, the true change and both ConvLSTM versions (seed 0):

In [ ]:
n = test_idx[len(test_idx) // 2]
w = windows[n]
panels = [(change[n], 'true change'),
          (predict(models['maps only', 0], np.array([n]))[0], 'maps only'),
          (predict(models['maps + drivers', 0], np.array([n]))[0], 'maps + drivers')]

fig, axes = plt.subplots(1, 3, figsize=(10, 6))
for ax, (img, title) in zip(axes, panels):
    shown = ax.imshow(np.where(masks[n], img, np.nan), origin='lower', cmap='RdBu_r', vmin=-3, vmax=3)
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(shown, ax=axes, shrink=0.6, label='ft (+ = deeper)')
fig.suptitle(f'{dates[w[-2]]} → {dates[w[-1]]}')
plt.show()